In [1]:
import os
import pandas as pd

# Use GPU 2
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["WANDB_MODE"] = "disabled"

import torch


from datasets import Dataset
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments


# Settings
MODEL_NAME = 'roberta-base'
DATA_FP = '../../annotation/email_understanding/annotation_output/full/df4model.csv'

2024-03-23 14:51:10.362274: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


**Read and prepare the data**

In [2]:
df = pd.read_csv(DATA_FP, index_col=0)
df = df.drop('batch', axis=1)
df = df.rename({'cleaned_text': 'text'}, axis=1)


def preprocess_data(examples):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenized_inputs = tokenizer(examples['text'], padding="max_length", max_length=512, truncation=True)
    
    # Extracting the labels from the example
    labels = [
        examples[label] for label in examples.keys() 
        if label.startswith('label_')
    ]
    # Transpose to get a list of lists where each inner list represents one example's labels
    labels = list(map(list, zip(*labels)))
    tokenized_inputs["labels"] = labels
    
    return tokenized_inputs


train = df[df['split'] == 'train'].drop('split', axis=1)
dev = df[df['split'] == 'val'].drop('split', axis=1)
test = df[df['split'] == 'test'].drop('split', axis=1)

train_dataset = Dataset.from_pandas(train)
dev_dataset = Dataset.from_pandas(dev)
test_dataset = Dataset.from_pandas(test)

train_dataset = train_dataset.map(preprocess_data, batched=True)
dev_dataset = dev_dataset.map(preprocess_data, batched=True)
test_dataset = test_dataset.map(preprocess_data, batched=True)

# Make sure there are no empty labels
train_dataset = train_dataset.filter(lambda example: None not in example['labels'])
dev_dataset = dev_dataset.filter(lambda example: None not in example['labels'])
test_dataset = test_dataset.filter(lambda example: None not in example['labels'])


print(f"Train length: {len(train_dataset)}. Eval length: {len(dev_dataset)}. Test length: {len(test_dataset)}")
train.head(3)

Map:   0%|          | 0/1149 [00:00<?, ? examples/s]

Map:   0%|          | 0/162 [00:00<?, ? examples/s]

Map:   0%|          | 0/329 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1149 [00:00<?, ? examples/s]

Filter:   0%|          | 0/162 [00:00<?, ? examples/s]

Filter:   0%|          | 0/329 [00:00<?, ? examples/s]

Train length: 1149. Eval length: 162. Test length: 329


,instance_id,label_Act:::Thank you/Welcome,label_Act:::Commit/Agree,label_Act:::Request,label_Act:::Deliver/Informative,label_Act:::Other_merged,label_spam,label_Sender Expectation,root_id,message_id,text
1,1_1,0,0,0,1,0,0,0.0,<GHr1h5kIkXUNFsvQ@example.com>,<UoYuNFi5oDmtfWb6@example.com>,Re: need help fixing broken tests in py3k-pep3...
2,1_10,1,0,1,0,0,0,1.0,<ck7qxRTi658O1J07@example.com>,<ck7qxRTi658O1J07@example.com>,errors using mdbus commands \n I'm having some...
3,1_11,0,0,0,1,0,0,1.0,<kVC09ThuoFIr8ckf@example.com>,<BWrL8J9wn9HNi4OH@example.com>,Re: Adding test case to test run gives error \...


**Setup the pretrained model and datasets**

In [3]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=7)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
training_args = TrainingArguments(
    output_dir=f'/shared/4/projects/research-jam-2024/models/fine-tuned/{MODEL_NAME}/',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    evaluation_strategy="epoch",  # Evaluate at the end of each epoch
    logging_dir='./logs',
    save_strategy="epoch",  # Save a model checkpoint at the end of each epoch
    save_total_limit=1,  # Only keep the last checkpoint
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model="eval_loss",  # Use eval_loss to identify the best model
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
)

# Train the model
trainer.train()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Epoch,Training Loss,Validation Loss
1,No log,0.355498
2,0.407900,0.330124
3,0.407900,0.292646


Checkpoint destination directory /shared/4/projects/research-jam-2024/models/fine-tuned/roberta-base/checkpoint-576 already exists and is non-empty. Saving will proceed but saved results may be invalid.


TrainOutput(global_step=864, training_loss=0.35961587340743456, metrics={'train_runtime': 117.1545, 'train_samples_per_second': 29.423, 'train_steps_per_second': 7.375, 'total_flos': 906984523238400.0, 'train_loss': 0.35961587340743456, 'epoch': 3.0})

**Evaluate on the test set**

In [5]:
# Evaluate the model on the test set
predictions, labels, _ = trainer.predict(test_dataset)
predictions = torch.sigmoid(torch.tensor(predictions)).numpy() > 0.5

labels = test_dataset["labels"]
predictions = predictions.astype(int)

# Generating classification report for each label
label_names = [
        'label_Act:::Thank you/Welcome', 'label_Act:::Commit/Agree', 'label_Act:::Request',
        'label_Act:::Deliver/Informative', 'label_Act:::Other_merged',
        'label_spam', 'label_Sender Expectation'
]
print("RoBERTa Base results")
print(classification_report(labels, predictions, target_names=label_names))

RoBERTa Base results
                                 precision    recall  f1-score   support

  label_Act:::Thank you/Welcome       0.72      0.54      0.62        24
       label_Act:::Commit/Agree       0.00      0.00      0.00        22
            label_Act:::Request       0.79      0.86      0.83       130
label_Act:::Deliver/Informative       0.86      0.81      0.83       219
       label_Act:::Other_merged       0.00      0.00      0.00        28
                     label_spam       0.71      0.50      0.59        10
       label_Sender Expectation       0.80      0.85      0.82       137

                      micro avg       0.82      0.74      0.78       570
                      macro avg       0.56      0.51      0.53       570
                   weighted avg       0.74      0.74      0.74       570
                    samples avg       0.84      0.77      0.78       570



/opt/anaconda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


**Save the results to the server**

In [6]:
output_file = f'/shared/4/projects/research-jam-2024/models/fine-tuned/{MODEL_NAME}/predictions.csv'

In [7]:
predictions_binary = (torch.sigmoid(torch.tensor(predictions)).numpy() > 0.5).astype(int)

# Create a DataFrame with the instance_id from the test set
test_instance_ids = test['instance_id'].reset_index(drop=True)
predictions_df = pd.DataFrame(predictions_binary, columns=label_names)

# Combine instance IDs with predictions
final_predictions_df = pd.concat([test_instance_ids, predictions_df], axis=1)
final_predictions_df.to_csv(output_file, index=False)

print(f"Predictions saved to {output_file}")

Predictions saved to /shared/4/projects/research-jam-2024/models/fine-tuned/roberta-base/predictions.csv
